In [50]:
import numpy as np
from scipy import fft
from scipy.special import jv
from scipy.integrate import quad

a, b = 1000, 2000

def bessel_product_integral(f, mu, nu, beta, rmin=1e-6, rmax=1e6, n=4096, bias=0.0):
    """I(k) = ∫_0^∞ f(r) J_mu(k r) J_nu(beta r) dr on a log-spaced k grid."""
    dln = np.log(rmax / rmin) / n                        # log spacing
    r = np.exp(np.log(rmin) + (np.arange(n) + 0.5) * dln)
    rc = np.sqrt(r[0] * r[-1])

    offset = fft.fhtoffset(dln, mu=mu, bias=bias)        # low-ringing k grid
    k = np.exp(offset) / rc * np.exp((np.arange(n) - (n - 1) / 2) * dln)

    a = f(r) * jv(nu, beta * r)
    I = fft.fht(a, dln, mu=mu, offset=offset, bias=bias) / k
    return k, I

# Example: f(r) = r exp(-r^2), mu = nu = 1, beta = 2
f = lambda r: r * np.exp(-r**2/100_000)
k, I = bessel_product_integral(f, mu=75, nu=75, beta=82.0, rmin=a, rmax=b, n=4096*256)

# Values at your own frequencies: interpolate in log k
k_wanted = np.array([0.5, 1.0, 3.0, 100.0])
I_wanted = np.interp(np.log(k_wanted), np.log(k), I)

In [35]:
%timeit bessel_product_integral(f, mu=75, nu=75, beta=82.0, rmin=a, rmax=b, n=4096)

1e+03 μs ± 5.43 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [51]:
I_wanted[-1]

np.float64(-0.0008287147033969823)

In [52]:
quad(lambda r: f(r)*jv(75,k_wanted[-1]*r)*jv(75,82.0*r), a, b, limit=100000, epsabs=1e-35)

(8.396447625300336e-09, 1.2504300695214264e-16)